# Lesson 08 Lab — BatchNorm Scale Factors and Network Slimming

**Puzzle:** When does a small BatchNorm gamma become a removable channel rather than merely a small multiplier?

This notebook is designed for a CUDA GPU and retains the output of a complete RTX 5090 run.


## Why this matters

Network Slimming creates a train-time ranking signal by regularizing BatchNorm scale factors. The scale does not remove a channel by itself. Deployment still requires selecting indices, rebuilding the producing convolution, slicing BatchNorm state, and propagating the same indices into every consumer.


## 0. Predict before running

1. Predict whether simply zeroing gamma matches physical deletion when beta is nonzero.
2. List every BatchNorm tensor that must be sliced.
3. Predict the output drift between a properly masked control and narrowed model.

For every answer, name the observation that would prove it wrong.


## 1. Name the concrete objects

A Conv-BN-ReLU-Conv block supplies convolution filters, BatchNorm gamma/beta/running statistics, retained channel indices, a gamma-masked control, and a physically narrowed copy.

- Gamma is an importance signal, not a structural deletion.
- BatchNorm affine and running-state tensors share the channel axis.
- Consumer weights must receive the identical retained indices.


## 2. Derive the mechanism

BatchNorm output per channel is `y_c = gamma_c (x_c - mu_c)/sqrt(var_c+eps) + beta_c`. A small gamma suppresses normalized variation, but beta can still contribute a constant and downstream weights can amplify it. Ranking by `|gamma|` is therefore a pruning heuristic learned under a sparsity regularizer. Physical removal is valid only when the chosen channel and all coupled parameters are sliced consistently and the resulting function is evaluated.

Keep value sparsity, physical shape, representation, and runtime evidence separate.


## 3. Verify the execution environment

Inspect the next cell before running it: it asserts CUDA, fixes the seed, defines transparent timing/numerical helpers, and prints the GPU/PyTorch/CUDA record needed to interpret every output.


In [1]:
LESSON_NO = 8
LESSON_TITLE = 'BatchNorm Scale Factors and Network Slimming'

from pathlib import Path
import copy, gzip, hashlib, importlib.util, io, json, math, random, shutil, statistics, sys
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), "This lab requires a CUDA-capable GPU."
DEVICE = torch.device("cuda")
SEED = 20260808 + LESSON_NO
random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cuda.matmul.allow_tf32 = False

gpu_name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
ENV = {
    "gpu": gpu_name,
    "compute_capability": f"{major}.{minor}",
    "torch": torch.__version__,
    "cuda_runtime": str(torch.version.cuda),
    "python": sys.version.split()[0],
    "seed": SEED,
}
print(json.dumps(ENV, indent=2))

def percentile(values, q):
    ordered = sorted(float(v) for v in values)
    if not ordered:
        return float("nan")
    position = (len(ordered) - 1) * q
    lo, hi = math.floor(position), math.ceil(position)
    if lo == hi:
        return ordered[lo]
    return ordered[lo] * (hi - position) + ordered[hi] * (position - lo)

def cuda_times(fn, warmup=6, repeats=24):
    with torch.inference_mode():
        for _ in range(warmup):
            fn()
        torch.cuda.synchronize()
        samples = []
        for _ in range(repeats):
            start = torch.cuda.Event(enable_timing=True)
            end = torch.cuda.Event(enable_timing=True)
            start.record()
            fn()
            end.record()
            end.synchronize()
            samples.append(float(start.elapsed_time(end)))
    return samples

def timing_summary(samples):
    return {
        "median_ms": float(statistics.median(samples)),
        "p95_ms": float(percentile(samples, 0.95)),
        "p99_ms": float(percentile(samples, 0.99)),
        "samples_ms": [float(x) for x in samples],
    }

def count_params(module):
    return int(sum(p.numel() for p in module.parameters()))

def zero_fraction(tensor):
    return float((tensor == 0).float().mean().item())

def magnitude_mask(tensor, sparsity):
    flat = tensor.detach().abs().flatten()
    prune_count = int(round(flat.numel() * float(sparsity)))
    prune_count = min(max(prune_count, 0), flat.numel())
    mask = torch.ones_like(flat)
    if prune_count:
        idx = torch.topk(flat, prune_count, largest=False).indices
        mask[idx] = 0
    return mask.view_as(tensor)

def exact_2_4_mask(weight):
    assert weight.shape[-1] % 4 == 0
    groups = weight.detach().abs().reshape(*weight.shape[:-1], -1, 4)
    keep = torch.topk(groups, 2, dim=-1, largest=True).indices
    mask = torch.zeros_like(groups)
    mask.scatter_(-1, keep, 1)
    return mask.reshape_as(weight)

def compliance_2_4(weight):
    groups = weight.detach().reshape(*weight.shape[:-1], -1, 4)
    return float(((groups != 0).sum(dim=-1) == 2).float().mean().item())

def tensor_metrics(reference, candidate):
    ref = reference.float()
    cand = candidate.float()
    delta = cand - ref
    return {
        "rmse": float(torch.sqrt(torch.mean(delta.square())).item()),
        "mae": float(torch.mean(delta.abs()).item()),
        "max_error": float(delta.abs().max().item()),
        "cosine": float(F.cosine_similarity(ref.flatten(), cand.flatten(), dim=0).item()),
    }

def spearman(a, b):
    a = torch.as_tensor(a, dtype=torch.float64)
    b = torch.as_tensor(b, dtype=torch.float64)
    ra = torch.empty_like(a)
    rb = torch.empty_like(b)
    ra[torch.argsort(a)] = torch.arange(a.numel(), dtype=torch.float64)
    rb[torch.argsort(b)] = torch.arange(b.numel(), dtype=torch.float64)
    ra -= ra.mean(); rb -= rb.mean()
    return float((ra @ rb / (ra.norm() * rb.norm() + 1e-12)).item())


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "torch": "2.12.0",
  "cuda_runtime": "13.0",
  "python": "3.12.13",
  "seed": 20260816
}


## 4. Freeze the comparison

| Role | Frozen value |
|---|---|
| Baseline | gamma-masked full-width Conv-BN-ReLU-Conv block |
| Candidate | physically narrowed block using the same retained gamma-ranked channels |
| Held constant | input, retained indices, all copied Conv/BN parameters, eval mode, dtype, and timing protocol |
| Measurements | gamma threshold, retained channels, output max error, parameters, and median latency |
| Evidence | `numerical-model` |

**Experiment:** Rank channels by gamma, create a semantics-preserving masked control, and rebuild the block at half width.


## 5. Read the experiment code

The notebook sets the removed channels to a neutral post-BN value in the control before copying the retained convolution filters, BN state, and second-layer input slices. Eval mode freezes running statistics. The equivalence check isolates structural bookkeeping from the separate question of whether gamma ranking preserves task quality.

Do not execute until the code implements the frozen table above.


In [2]:
class SlimBlock(nn.Module):
    def __init__(self, channels=24):
        super().__init__(); self.conv1=nn.Conv2d(8,channels,3,padding=1,bias=False); self.bn=nn.BatchNorm2d(channels); self.conv2=nn.Conv2d(channels,12,3,padding=1,bias=False)
    def forward(self,x): return self.conv2(F.relu(self.bn(self.conv1(x))))

full = SlimBlock().to(DEVICE).eval()
with torch.no_grad():
    full.bn.weight.copy_(torch.linspace(0.02, 1.2, 24, device=DEVICE)[torch.randperm(24, device=DEVICE)])
    full.bn.bias.zero_(); full.bn.running_mean.zero_(); full.bn.running_var.fill_(1)
keep = torch.topk(full.bn.weight.abs(), 12).indices.sort().values
remove_mask = torch.ones(24, dtype=torch.bool, device=DEVICE); remove_mask[keep] = False
masked = copy.deepcopy(full)
with torch.no_grad(): masked.bn.weight[remove_mask] = 0; masked.bn.bias[remove_mask] = 0
narrow = SlimBlock(12).to(DEVICE).eval()
with torch.no_grad():
    narrow.conv1.weight.copy_(full.conv1.weight[keep]); narrow.bn.weight.copy_(full.bn.weight[keep]); narrow.bn.bias.copy_(full.bn.bias[keep])
    narrow.bn.running_mean.copy_(full.bn.running_mean[keep]); narrow.bn.running_var.copy_(full.bn.running_var[keep]); narrow.conv2.weight.copy_(full.conv2.weight[:,keep])
x = torch.randn(12,8,32,32,device=DEVICE)
with torch.inference_mode(): ym, yn = masked(x), narrow(x)
tm=timing_summary(cuda_times(lambda: masked(x))); tn=timing_summary(cuda_times(lambda: narrow(x)))
metrics={
    "retained_channels": int(keep.numel()), "gamma_threshold": float(full.bn.weight[keep].abs().min().item()),
    "max_error": float((ym-yn).abs().max().item()), "full_parameters": count_params(full), "narrow_parameters": count_params(narrow),
    "masked_median_ms": tm["median_ms"], "narrow_median_ms": tn["median_ms"], "kept_indices": keep.tolist(),
}
analysis=(
    f"The gamma ranking retained {metrics['retained_channels']} channels above an absolute threshold of "
    f"{metrics['gamma_threshold']:.6f}. After slicing convolution and every BatchNorm state tensor, the narrow "
    f"output matched the gamma-masked control within {metrics['max_error']:.3e}. Parameters fell from "
    f"{metrics['full_parameters']:,} to {metrics['narrow_parameters']:,}; ranking quality on a real task remains unmeasured."
)


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| Retained channels | 12 |
| Gamma threshold | 0.635652 |
| Output max error | 0.000244 |
| Full parameters | 4,368 |
| Narrow parameters | 2,184 |
| Narrow median | 0.044560 ms |


## 7. Interpret rather than merely print

The gamma ranking retained 12 channels above an absolute threshold of 0.635652. After slicing convolution and every BatchNorm state tensor, the narrow output matched the gamma-masked control within 2.438e-04. Parameters fell from 4,368 to 2,184; ranking quality on a real task remains unmeasured.

The result is bounded to the shapes, seed, packages, and evidence label printed here.


## 8. Keep the evidence label honest

This run is labeled **`numerical-model`**. The CUDA experiment isolates a numerical mechanism. It is not a full paper reproduction, trained production model, or native sparse-kernel benchmark.

The next cell writes the canonical JSON artifact and prints the same payload.


In [3]:
artifact = Path("artifacts/rtx5090-result.json")
artifact.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "lesson": 8,
    "title": 'BatchNorm Scale Factors and Network Slimming',
    "environment": ENV,
    "evidence_label": 'numerical-model',
    "metrics": metrics,
    "analysis": analysis,
    "conclusion": 'Network Slimming turns BatchNorm scales into a ranking mechanism; deployment benefit begins only after consistent structural removal.',
}
artifact.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
print(json.dumps(payload, indent=2, ensure_ascii=False))


{
  "lesson": 8,
  "title": "BatchNorm Scale Factors and Network Slimming",
  "environment": {
    "gpu": "NVIDIA GeForce RTX 5090",
    "compute_capability": "12.0",
    "torch": "2.12.0",
    "cuda_runtime": "13.0",
    "python": "3.12.13",
    "seed": 20260816
  },
  "evidence_label": "numerical-model",
  "metrics": {
    "retained_channels": 12,
    "gamma_threshold": 0.6356521844863892,
    "max_error": 0.00024378299713134766,
    "full_parameters": 4368,
    "narrow_parameters": 2184,
    "masked_median_ms": 0.056463999673724174,
    "narrow_median_ms": 0.04456000030040741,
    "kept_indices": [
      0,
      2,
      4,
      5,
      11,
      15,
      18,
      19,
      20,
      21,
      22,
      23
    ]
  },
  "analysis": "The gamma ranking retained 12 channels above an absolute threshold of 0.635652. After slicing convolution and every BatchNorm state tensor, the narrow output matched the gamma-masked control within 2.438e-04. Parameters fell from 4,368 to 2,184; rank

## 9. Make the bounded decision

> Network Slimming turns BatchNorm scales into a ranking mechanism; deployment benefit begins only after consistent structural removal.

**Acceptance/rollback:** Accept the ranking only after held-out quality, coupled slicing, physical width, and runtime evidence all pass.

**Failure analysis:** Small gamma values can be scale-invariant with neighboring weights, and nonzero beta breaks naive zero-gamma reasoning. Training without the intended L1 pressure may produce an uninformative ranking. Residual and concatenation consumers need a dependency graph beyond this local block.


## 10. Extend the evidence

Train gamma with an explicit sparsity penalty, compare rankings across seeds, and propagate selected channels through a residual model with a graph-level pruning tool.

The full evidence boundary and references are in [`README.md`](README.md).
